In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

X_train = pd.read_csv('../data/processed/X_train.csv')
feature_names = X_train.columns.tolist()


with open('../models/shap_values_dict.pkl', 'rb') as f:
    shap_values_dict = pickle.load(f)
with open('../models/shap_local_dict.pkl', 'rb') as f:
    shap_local_dict = pickle.load(f)
with open('../models/lime_results.pkl', 'rb') as f:
    lime_results = pickle.load(f)
with open('../models/perm_results.pkl', 'rb') as f:
    perm_results = pickle.load(f)
with open('../models/pdp_results.pkl', 'rb') as f:
    pdp_results = pickle.load(f)
with open('../models/dice_results.pkl', 'rb') as f:
    dice_results = pickle.load(f)

models_names = ['Random Forest', 'Decision Tree', 'DNN']

In [53]:
def build_xai_context(model_name, obs_name):
    
    
    shap_local = shap_local_dict[model_name][obs_name]
    sv = shap_local['shap_values']
    data = shap_local['data']
    prediction = shap_local['prediction']
    
    pred_class = 'Attaque' if prediction[1] > 0.5 else 'Normal'
    confidence = max(prediction[0], prediction[1]) * 100
    
    sorted_idx = np.argsort(np.abs(sv))[::-1][:3]
    shap_local_top = [(feature_names[i], float(sv[i]), float(data[i])) 
                      for i in sorted_idx]
    
    
    lime_data = lime_results[model_name][obs_name]
    lime_top = sorted(lime_data['features'], key=lambda x: abs(x[1]), reverse=True)[:3]
    
    
    shap_global = shap_values_dict[model_name]
    if isinstance(shap_global, list):
        imp_global = np.abs(shap_global[1]).mean(axis=0)
    else:
        imp_global = np.abs(shap_global).mean(axis=0)
    sorted_global = np.argsort(imp_global)[::-1][:3]
    shap_global_top = [(feature_names[i], float(imp_global[i])) 
                       for i in sorted_global]
    
    
    perm = perm_results[model_name]
    imp_perm = np.array(perm['importances_mean'])
    sorted_perm = np.argsort(imp_perm)[::-1][:3]
    perm_top = [(feature_names[i], float(imp_perm[i])) 
                for i in sorted_perm]
    
    
    pdp = pdp_results[model_name]
    pdp_features = pdp['features']
    pdp_ranges = []
    for i, (avg, grid) in enumerate(pdp['pd_results']):
        avg_flat = avg.flatten()  # Convertir en 1D
        grid_flat = grid[0]       # Prendre le premier array de la liste
        pdp_ranges.append((
            pdp_features[i], 
            float(np.min(grid_flat)), 
            float(np.max(grid_flat)), 
            float(avg_flat[np.argmax(avg_flat)])
        ))
    
   
    dice_cf = dice_results[model_name][obs_name]
    try:
        cf_df = dice_cf.cf_examples_list[0].final_cfs_df
        original = dice_cf.cf_examples_list[0].test_instance_df
        changes = []
        for feat in feature_names:
            orig_val = float(original[feat].values[0])
            cf_val = float(cf_df[feat].iloc[0])
            if abs(orig_val - cf_val) > 0.01:
                changes.append((feat, orig_val, cf_val))
        changes = changes[:3]
    except:
        changes = []
    
    return {
        'pred_class': pred_class,
        'confidence': confidence,
        'shap_local_top': shap_local_top,
        'lime_top': lime_top,
        'shap_global_top': shap_global_top,
        'perm_top': perm_top,
        'pdp_ranges': pdp_ranges,
        'dice_changes': changes
    }

In [54]:
feature_descriptions = {
    'spdx': 'vitesse longitudinale',
    'spdy': 'vitesse latérale',
    'spdx_n': 'vitesse longitudinale bruitée',
    'spdy_n': 'vitesse latérale bruitée',
    'posx': 'position longitudinale',
    'posy': 'position latérale',
    'posx_n': 'position longitudinale bruitée',
    'posy_n': 'position latérale bruitée',
    'aclx': 'accélération longitudinale',
    'acly': 'accélération latérale',
    'aclx_n': 'accélération longitudinale bruitée',
    'acly_n': 'accélération latérale bruitée',
    'hedx': 'direction longitudinale',
    'hedy': 'direction latérale',
    'hedx_n': 'direction longitudinale bruitée',
    'hedy_n': 'direction latérale bruitée'
}

def generate_explanation(model_name, obs_name, ctx, user_profile):

    if user_profile in ['conducteur', 'gestionnaire']:
        cas_description = "Le système a détecté une anomalie dans le comportement d'un véhicule proche" if ctx['pred_class'] == 'Attaque' else "Le système n'a détecté aucune anomalie"
    else:
        cas_description = f"Cas : {obs_name.replace('_', ' ')}"

    shap_features_desc = []
    for f, c, v in ctx['shap_local_top']:
        direction = "anormalement élevée" if v > 0.5 else "anormalement basse" if v < -0.5 else "légèrement anormale"
        sens_contribution = "pousse la prédiction vers Attaque" if c > 0 else "pousse la prédiction vers Normal"
        desc = feature_descriptions.get(f, f)
        shap_features_desc.append(f"- {desc} : valeur {direction}, et cette valeur {sens_contribution} (contribution SHAP {c:+.4f})")

    analyste_context = f"""
{cas_description}
Confiance numérique du système : {ctx['confidence']:.1f}%

Comportements anormaux détectés :
{chr(10).join([f"- {feature_descriptions.get(f, f)} : {('anormalement élevée' if v > 0.5 else 'anormalement basse' if v < -0.5 else 'légèrement anormale')} — {'contribue à l anomalie' if c > 0 else 'contribue à la normalité'}" for f, c, v in ctx['shap_local_top']])}

Confirmation par une deuxième méthode d'analyse :
{chr(10).join([f"- {feature_descriptions.get(f, f)} : {'contribue à l anomalie' if c > 0 else 'contribue à la normalité'}" for f, c in ctx['lime_top']])}

Comportements généralement les plus discriminants pour ce type de détection :
{chr(10).join([f"- {feature_descriptions.get(f, f)}" for f, i in ctx['shap_global_top']])}

Pour que le système change sa décision, il faudrait que :
{chr(10).join([f"- la {feature_descriptions.get(f, f)} change de valeur" for f, o, c in ctx['dice_changes']]) if ctx['dice_changes'] else "Aucun changement minimal identifié"}
"""

    xai_context = f"""
Modèle : {model_name}
{cas_description}
Confiance numérique de la prédiction : {ctx['confidence']:.1f}%

Principaux indicateurs SHAP Local :
{chr(10).join(shap_features_desc)}

Confirmation LIME :
{chr(10).join([f"- {feature_descriptions.get(f, f)} : {'pousse vers Attaque' if c > 0 else 'pousse vers Normal'} (contribution {c:+.4f})" for f, c in ctx['lime_top']])}

Features globalement importantes (SHAP Global) :
{chr(10).join([f"- {feature_descriptions.get(f, f)} : importance {i:.4f}" for f, i in ctx['shap_global_top']])}

Validation indépendante (Permutation Importance) :
{chr(10).join([f"- {feature_descriptions.get(f, f)} : importance {i:.4f}" for f, i in ctx['perm_top']])}

Seuils critiques (PDP) :
{chr(10).join([f"- {feature_descriptions.get(f, f)} : entre {mn:.2f} et {mx:.2f}" for f, mn, mx, pk in ctx['pdp_ranges'][:3]])}

Pour changer la prédiction (DICE) :
{chr(10).join([f"- {feature_descriptions.get(f, f)} devrait changer de {o:.3f} vers {c:.3f}" for f, o, c in ctx['dice_changes']]) if ctx['dice_changes'] else "Aucun changement minimal identifié"}
"""

    profiles = {
        'conducteur': f"""Tu es un système d'alerte de sécurité embarqué dans un véhicule connecté VANET.
Tu dois alerter le conducteur d'un danger potentiel détecté par le système IDS.

Règles STRICTES :
- Ne mentionne JAMAIS de valeurs numériques, pourcentages ou termes techniques
- Ne décris JAMAIS une scène physique inventée (zigzague, accélère, freine...)
- Le système détecte des incohérences dans les messages transmis, pas des comportements visuels
- Message sobre et factuel : un véhicule transmet des informations incohérentes
- Donne UNE action claire et immédiate au conducteur
- Maximum 2 phrases courtes
- Style : message d'alerte embarqué, direct et sobre

Contexte :
{cas_description}
{chr(10).join(shap_features_desc)}

Exemple de bon message : "Un véhicule voisin transmet des informations incohérentes. Restez vigilant et augmentez la distance de sécurité."
""",

        'gestionnaire': f"""Tu es un assistant pour un gestionnaire d'un centre de supervision VANET.
Il supervise les communications véhiculaires et la sécurité du réseau VANET.

Règles STRICTES :
- Ne mentionne PAS de méthodes d'analyse (SHAP, LIME, PDP, DICE, IA, modèle)
- Ne mentionne PAS de noms de features techniques (spdx, spdy, posx...)
- Utilise le vocabulaire VANET : messages coopératifs, BSM, falsification, isolation, véhicule suspect
- Ne donne JAMAIS de chiffres précis sur la distance (pas de "500 mètres") — utilise "dans le voisinage" ou "dans la zone locale"
- Décris uniquement les comportements anormaux fournis, sans en déduire d'impact non mentionné
- Propose UNIQUEMENT l'action d'isoler les messages de ce véhicule — n'invente aucune autre action
- Maximum 2 phrases

Contexte :
{cas_description}
{chr(10).join([f"- {feature_descriptions.get(f, f)} anormale" for f, c, v in ctx['shap_local_top'] if c > 0])}

Exemple de bon message : "Un véhicule a été identifié comme source probable de messages BSM falsifiés. Recommandation : isoler les messages provenant de ce véhicule."
""",

        'analyste': f"""Tu es un assistant pour un analyste en cybersécurité dans un centre de supervision SOC-VANET.
L'analyste est expert en sécurité des réseaux véhiculaires mais ne connaît pas les méthodes d'IA ni les noms des variables techniques du modèle.

Règles STRICTES :
- Ne mentionne JAMAIS les noms de méthodes XAI (SHAP, LIME, PDP, DICE, Permutation Importance)
- Ne mentionne JAMAIS les noms de features techniques (spdx, spdy, posx, posy, aclx, acly, hedx, hedy...)
- Ne mentionne JAMAIS d'actions d'ingénierie (ajuster les seuils, recalibrer le modèle, modifier les poids...)
- Ne propose AUCUNE action de surveillance, de vérification ou opérationnelle — les données XAI ne permettent pas de déduire une action, contente-toi d'expliquer la décision
- Pour DICE : présente les contrefactuels comme "L'analyse contrefactuelle indique qu'une variation de [comportement] aurait pu modifier la décision du système"
- Utilise UNIQUEMENT la valeur de confiance numérique fournie
- Reste strictement factuel — ne déduis rien qui ne soit pas explicitement dans le contexte fourni
- Pour les faux négatifs : utilise "L'analyse a posteriori révèle qu'une attaque réelle n'a pas été détectée"
- Maximum 4 phrases

Structure de la réponse :
1. Niveau de confiance du système (reprendre le pourcentage fourni)
2. Comportements anormaux détectés
3. Information contrefactuelle si disponible

Contexte :
{analyste_context}
""",

        'ingenieur': f"""Tu es un assistant pour un ingénieur automobile spécialiste des systèmes IDS embarqués.
Il connaît parfaitement SHAP, LIME, PDP, DICE et les variables techniques du modèle.

Règles STRICTES :
- Utilise le vocabulaire technique complet (SHAP, LIME, features, valeurs numériques)
- Analyse le comportement technique du système IDS en te basant UNIQUEMENT sur les indicateurs fournis dans le contexte
- Pour les faux négatifs UNIQUEMENT : commence par "L'analyse a posteriori révèle qu'une attaque réelle n'a pas été détectée"
- Pour les vrais négatifs : commence par "Le système a correctement classifié ce message comme Normal"
- Pour les faux positifs : commence par "Le système a incorrectement classifié ce message Normal comme une Attaque"
- Pour les vrais positifs : commence par "Le système a correctement détecté une attaque"
- Identifie les causes possibles de la décision ou de l'erreur de classification, en te basant uniquement sur les indicateurs fournis dans le contexte
- Présente DICE uniquement comme une information contrefactuelle sur l'observation ("si [feature] avait été [valeur] au lieu de [valeur observée], le système aurait classifié différemment"), jamais comme une action à effectuer sur le système
- N'invente aucune technique (clustering, pondération adaptative, réévaluation de seuils non chiffrés...) qui n'apparaît pas explicitement dans le contexte
- Maximum 4 phrases techniques précises

Contexte :
{xai_context}
"""
    }
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": profiles[user_profile]}],
        max_tokens=300,
        temperature=0.3
    )
    
    return response.choices[0].message.content

In [55]:
profile_cases = {
    'conducteur': ['Vrai_Positif'],
    'gestionnaire': ['Vrai_Positif'],
    'analyste': ['Vrai_Positif', 'Faux_Positif', 'Faux_Negatif'],
    'ingenieur': ['Vrai_Positif', 'Vrai_Negatif', 'Faux_Positif', 'Faux_Negatif']
}

profile_names = {
    'conducteur': 'Conducteur du véhicule',
    'gestionnaire': 'Gestionnaire du trafic',
    'analyste': 'Analyste en cybersécurité',
    'ingenieur': 'Ingénieur automobile'
}

llm_results = {}

for model_name in models_names:
    llm_results[model_name] = {}
    print(f'\n{"="*50}')
    print(f'Modèle : {model_name}')
    print(f'{"="*50}')
    
    for obs_name in ['Vrai_Positif', 'Vrai_Negatif', 'Faux_Positif', 'Faux_Negatif']:
        llm_results[model_name][obs_name] = {}
        
        ctx = build_xai_context(model_name, obs_name)
        
        print(f'\n--- {obs_name} ---')
        
        for profile, cases in profile_cases.items():
            if obs_name in cases:
                explanation = generate_explanation(model_name, obs_name, ctx, profile)
                llm_results[model_name][obs_name][profile] = explanation
                print(f'\n[{profile_names[profile]}]')
                print(explanation)


with open('../models/llm_results.pkl', 'wb') as f:
    pickle.dump(llm_results, f)



Modèle : Random Forest

--- Vrai_Positif ---

[Conducteur du véhicule]
Un véhicule voisin transmet des informations incohérentes. Restez vigilant et augmentez la distance de sécurité.

[Gestionnaire du trafic]
Un véhicule a été identifié comme source probable de messages BSM falsifiés en raison de vitesses longitudinales et latérales anormales. Recommandation : isoler les messages provenant de ce véhicule.

[Analyste en cybersécurité]
1. Niveau de confiance du système : 91.9%.  
2. Comportements anormaux détectés : la vitesse longitudinale et la vitesse latérale, ainsi que leur version bruitée, sont légèrement anormales et contribuent à l'anomalie.  
3. L'analyse contrefactuelle indique qu'une variation de l'accélération longitudinale ou de l'accélération latérale aurait pu modifier la décision du système.

[Ingénieur automobile]
Le système a correctement détecté une attaque avec une confiance de 91.9%. Les indicateurs SHAP locaux montrent que les valeurs légèrement anormales de la vi

In [56]:
import numpy as np
import pickle

with open('../models/llm_results.pkl', 'rb') as f:
    llm_results = pickle.load(f)

for model_name in models_names:
    print(f'\n{"="*50} {model_name} {"="*50}')
    
    for obs_name in shap_local_dict[model_name].keys():
        if obs_name not in llm_results[model_name] or not llm_results[model_name][obs_name]:
            continue
        
        pred = shap_local_dict[model_name][obs_name]['prediction']
        sv = shap_local_dict[model_name][obs_name]['shap_values']
        
        print(f'\n--- {obs_name} | Normal={pred[0]:.2f} | Attaque={pred[1]:.2f}')
        
        # SHAP Local
        print('Top 3 SHAP Local :')
        for i in np.argsort(np.abs(sv))[::-1][:3]:
            print(f'  {feature_names[i]:15s} : {sv[i]:+.4f} {"→ Attaque" if sv[i] > 0 else "→ Normal"}')
        
        # SHAP Global
        shap_global = shap_values_dict[model_name]
        imp_global = np.abs(shap_global[1] if isinstance(shap_global, list) else shap_global).mean(axis=0)
        print('Top 3 SHAP Global :')
        for i in np.argsort(imp_global)[::-1][:3]:
            print(f'  {feature_names[i]:15s} : {imp_global[i]:.4f}')
        
        # LIME
        lime_data = lime_results[model_name][obs_name]['features']
        print('Top 3 LIME :')
        for feat, val in sorted(lime_data, key=lambda x: abs(x[1]), reverse=True)[:3]:
            print(f'  {feat:30s} : {val:+.4f} {"→ Attaque" if val > 0 else "→ Normal"}')
        
        # Permutation Importance
        perm = perm_results[model_name]
        imp_perm = np.array(perm['importances_mean'])
        print('Top 3 Permutation Importance :')
        for i in np.argsort(imp_perm)[::-1][:3]:
            print(f'  {feature_names[i]:15s} : {imp_perm[i]:.4f}')
        
        # PDP
        pdp = pdp_results[model_name]
        pdp_imp = [np.max(avg.flatten()) - np.min(avg.flatten()) for avg, _ in pdp['pd_results']]
        print('Top 3 PDP (amplitude) :')
        for i in np.argsort(pdp_imp)[::-1][:3]:
            print(f'  {pdp["features"][i]:15s} : amplitude={pdp_imp[i]:.4f}')
        
        # DICE
        try:
            cf = dice_results[model_name][obs_name]
            cf_df = cf.cf_examples_list[0].final_cfs_df
            original = cf.cf_examples_list[0].test_instance_df
            changes = [(f, float(original[f].values[0]), float(cf_df[f].iloc[0]))
                      for f in feature_names if abs(float(cf_df[f].iloc[0]) - float(original[f].values[0])) > 0.01][:3]
            print('DICE — features à modifier :')
            for f, orig, new in changes:
                print(f'  {f:15s} : {orig:.3f} → {new:.3f}')
        except:
            print('DICE : non disponible')
        
        # Explications LLM
        for profile, expl in llm_results[model_name][obs_name].items():
            print(f'\n  [{profile}]\n  {expl}')


================================================== Random Forest ==================================================

--- Vrai_Negatif | Normal=0.88 | Attaque=0.12
Top 3 SHAP Local :
  spdx            : -0.0889 → Normal
  spdy            : -0.0739 → Normal
  posy            : -0.0438 → Normal
Top 3 SHAP Global :
  spdx            : 0.0882
  spdy            : 0.0660
  posy            : 0.0372
Top 3 LIME :
  -0.61 < spdy <= -0.23          : -0.0521 → Normal
  0.12 < posy <= 0.76            : -0.0396 → Normal
  -0.30 < aclx <= -0.08          : -0.0133 → Normal
Top 3 Permutation Importance :
  spdx            : 0.3510
  spdy            : 0.3226
  posx            : 0.2351
Top 3 PDP (amplitude) :
  posy            : amplitude=0.3598
  spdx            : amplitude=0.3439
  posx            : amplitude=0.2862
DICE — features à modifier :
  spdy_n          : -0.002 → -4.350
  hedx_n          : -0.835 → 6.912

  [ingenieur]
  Le système a correctement classifié ce message comme Normal avec une con

In [ ]:
import anthropic
import pickle

judge_client = anthropic.Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY')) 

In [58]:
with open('../models/llm_results.pkl', 'rb') as f:
    llm_results = pickle.load(f)

In [59]:
def get_reference_text(model_name, obs_name):
    ctx = build_xai_context(model_name, obs_name)
    
    label_reel = {
        'Vrai_Positif': "Attaque (le système a correctement prédit Attaque)",
        'Vrai_Negatif': "Normal (le système a correctement prédit Normal)",
        'Faux_Positif': "Normal (le système a INCORRECTEMENT prédit Attaque, c'est une fausse alerte)",
        'Faux_Negatif': "Attaque (le système a INCORRECTEMENT prédit Normal, l'attaque a été manquée)"
    }
    
    texte = f"""
Cas réel : {obs_name.replace('_', ' ')} — Vrai label : {label_reel[obs_name]}
Prédiction du modèle : {ctx['pred_class']} (confiance {ctx['confidence']:.1f}%)

SHAP Local (top 3) :
{chr(10).join([f"- {f} : valeur {v:.3f}, contribution {c:+.4f}" for f, c, v in ctx['shap_local_top']])}

SHAP Global (top 3) :
{chr(10).join([f"- {f} : importance {i:.4f}" for f, i in ctx['shap_global_top']])}

LIME (top 3) :
{chr(10).join([f"- {f} : contribution {c:+.4f}" for f, c in ctx['lime_top']])}

Permutation Importance (top 3) :
{chr(10).join([f"- {f} : importance {i:.4f}" for f, i in ctx['perm_top']])}

PDP (plages et pic) :
{chr(10).join([f"- {f} : entre {mn:.2f} et {mx:.2f}, pic à {pk:.2f}" for f, mn, mx, pk in ctx['pdp_ranges']])}

DICE (contrefactuels) :
{chr(10).join([f"- {f} : de {o:.3f} vers {n:.3f}" for f, o, n in ctx['dice_changes']]) if ctx['dice_changes'] else "Aucun"}
"""
    return texte

In [60]:
profile_descriptions = {
    'conducteur': "Conducteur du véhicule : utilisateur final, non expert en réseau ni en cybersécurité, orienté sécurité et confort. Tâches : conduite du véhicule, réception et interprétation des alertes de sécurité, réactions aux avertissements routiers. Besoin en explicabilité : comprendre pourquoi une alerte apparaît, le niveau de danger réel, l'action recommandée, avoir confiance dans le système.",
    
    'gestionnaire': "Gestionnaire du trafic routier (centre de gestion du trafic) : expert en transport et mobilité, non expert en cybersécurité. Tâches : supervision du trafic, gestion des feux intelligents, traitement des incidents, exploitation des données VANET. Besoin en explicabilité : comprendre pourquoi certains messages sont bloqués ou signalés comme faux, l'impact des décisions IDS sur la fluidité du trafic.",
    
    'analyste': "Analyste en cybersécurité (centre de supervision de sécurité) : expert en sécurité informatique et réseaux, mais non expert en intelligence artificielle. Tâches : surveillance des alertes IDS, analyse des attaques, identification des véhicules malveillants, prise de décision de mitigation. Besoin en explicabilité : explications détaillées en langage courant, un degré de confiance exprimé en pourcentage, et une justification des alertes formulée avec des comportements décrits en langage naturel, sans noms de méthodes XAI (SHAP, LIME...) ni valeurs numériques techniques brutes de contribution, qui ne lui sont pas compréhensibles." ,

    'ingenieur': "Ingénieur du constructeur automobile : expert en systèmes embarqués et logiciels automobiles. Tâches : intégration des systèmes VANET, validation des mécanismes de sécurité, maintenance logicielle. Besoin en explicabilité : comprendre le comportement du système IDS, les causes des erreurs, améliorer et corriger le système."
}

In [61]:
def build_judge_prompt(reference, profile, explanation):
    profile_desc = profile_descriptions[profile]
    
    return f"""Tu es un évaluateur indépendant. Ta tâche est de juger la qualité d'une explication générée par un système d'IA, en te basant UNIQUEMENT sur la référence ci-dessous.

RÉFÉRENCE (données réelles, ne juge que par rapport à cela) :
{reference}

PROFIL DESTINATAIRE DE L'EXPLICATION :
{profile_desc}

EXPLICATION À ÉVALUER :
{explanation}

Attribue 1 point pour chaque critère rempli, 0 sinon. Voici la grille précise pour chaque critère :

RAPPEL IMPORTANT SUR LES SIGNES : la convention "positif = pousse vers Attaque, négatif = pousse vers Normal" est ABSOLUE et ne dépend jamais du label réel du cas (Vrai Positif, Vrai Négatif, Faux Positif, Faux Négatif). Ne déduis JAMAIS une convention de signe différente à partir du label réel, vérifie uniquement le signe mathématique du chiffre donné dans la référence.

1. EXACTITUDE FACTUELLE
   - 0 : au moins une valeur numérique ou une direction (vers Attaque/Normal) est déformée ou inversée par rapport à la référence
   - 1 : toutes les valeurs et directions mentionnées correspondent exactement à la référence

2. ABSENCE D'INVENTION
   - 0 : l'explication mentionne une CAUSE ou une DONNÉE TECHNIQUE absente de la référence
   - 1 : l'explication ne contient que des informations directement déductibles de la référence
   NOTE IMPORTANTE : pour les profils conducteur et gestionnaire, une action de sécurité 
   générique standardisée (ex : "restez vigilant", "augmentez la distance de sécurité", 
   "isoler les messages de ce véhicule") ne doit PAS être comptée comme une invention, 
   car il s'agit d'une consigne opérationnelle imposée par système et non d'un fait déduit 
   par le LLM. Ne pénalise ce critère QUE si l'explication invente une cause technique, 
   une valeur chiffrée, ou une action non standard qui n'est pas dans la référence.
   NOTE COMPLÉMENTAIRE : la traduction d'un nom de feature technique en description physique 
   (ex : "spdx" → "vitesse longitudinale", "spdx_n" → "vitesse longitudinale bruitée") fait 
   partie du dictionnaire de correspondance fourni au système et n'est PAS une invention, 
   même si cette traduction n'apparaît pas littéralement dans la référence chiffrée.

3. ADAPTATION AU PROFIL
   - 0 : le vocabulaire ou le niveau de détail est inadapté au profil (jargon pour un non-expert, imprécision pour un expert), OU la majorité des besoins en explicabilité listés dans la description du profil sont absents de l'explication
   - 1 : le vocabulaire, le niveau de détail et le contenu répondent bien aux caractéristiques et aux besoins en explicabilité du profil décrit ci-dessus

4. CLARTÉ
   - 0 : l'explication est ambiguë, mal formulée ou difficile à suivre
   - 1 : l'explication est claire et compréhensible

IMPORTANT : ne tiens PAS compte de la longueur de l'explication dans ton jugement — une explication courte n'est pas moins bonne qu'une explication longue.

Tu DOIS impérativement suivre ces 2 étapes, dans cet ordre exact :

ÉTAPE 1 — Vérification signe par signe (obligatoire avant toute conclusion) :
Pour CHAQUE feature mentionnée dans l'explication avec une contribution chiffrée, vérifie 
individuellement et séparément :
- Feature : [nom]
- Contribution donnée dans la référence : [chiffre exact avec signe]
- Sens attendu selon la convention absolue : [Attaque si positif / Normal si négatif]
- Ce que dit l'explication : [reprend la formulation de l'explication]
- Concordance : [OUI / NON]

Fais cette vérification pour CHAQUE feature avant de passer à l'étape 2. Ne conclus à une 
erreur d'exactitude QUE si au moins une ligne de ce tableau indique "NON".

ÉTAPE 2 — Notation finale, basée UNIQUEMENT sur le tableau de l'étape 1 pour le critère Exactitude :

Justification : (résumé de 2-3 phrases basé sur le tableau ci-dessus)
Exactitude : (0 ou 1 — 1 uniquement si TOUTES les lignes du tableau indiquent "OUI")
Invention : (0 ou 1)
Profil : (0 ou 1)
Clarte : (0 ou 1)
Total : (somme des 4, entre 0 et 4)
"""

In [62]:
def judge_explanation(model_name, obs_name, profile, explanation):
    reference = get_reference_text(model_name, obs_name)
    prompt = build_judge_prompt(reference, profile, explanation)
    
    response = judge_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1200,
        temperature=0.1,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.content[0].text

In [63]:
evaluation_results = {}

for model_name in llm_results:
    evaluation_results[model_name] = {}
    for obs_name in llm_results[model_name]:
        evaluation_results[model_name][obs_name] = {}
        for profile, explanation in llm_results[model_name][obs_name].items():
            try:
                verdict = judge_explanation(model_name, obs_name, profile, explanation)
                evaluation_results[model_name][obs_name][profile] = verdict
                print(f'{model_name} | {obs_name} | {profile}')
                print(verdict)
                print('---')
            except Exception as e:
                print(f'ERREUR sur {model_name} | {obs_name} | {profile} : {e}')
                evaluation_results[model_name][obs_name][profile] = None

with open('../models/llm_evaluation.pkl', 'wb') as f:
    pickle.dump(evaluation_results, f)

Random Forest | Vrai_Positif | conducteur
# ÉTAPE 1 — Vérification signe par signe

L'explication évaluée ne mentionne **aucune feature spécifique avec contribution chiffrée**. Elle fournit uniquement une interprétation générale ("véhicule voisin transmet des informations incohérentes") et des consignes d'action ("restez vigilant et augmentez la distance de sécurité").

**Tableau de vérification :**
Aucune feature à vérifier individuellement, car l'explication ne cite aucune donnée technique ni contribution chiffrée.

**Conclusion pour l'exactitude :** Puisqu'aucune valeur numérique ou direction n'est mentionnée dans l'explication, il n'y a **aucune déformation ou inversion possible**. L'explication reste factuelle au niveau de la prédiction (Attaque détectée).

---

# ÉTAPE 2 — Notation finale

**Justification :**

L'explication ne mentionne aucune valeur numérique ni direction technique, donc aucune erreur factuelle n'est commise. Elle traduit la détection d'attaque en langage access